<!-- nav -->
[← 8 · C Types](08_07_C_Types.ipynb) · [Home](../../README.md) · [8.09 · Cpp Types →](08_09_Cpp_Types.ipynb)

# 8.08 · C Preprocessor

Before C or C++ reads a line of code, its preprocessor reads the directives among them: `#include <stdio.h>`,
`#define WIDTH 8`, `#if defined(X) && X > 1` ... `#endif` (ISO/IEC 9899, 6.10; ISO/IEC 14882, [cpp]). The two standards
define the same directives, and one dialect models them for both, `C Preprocessor`: its directives, `defined(X)`
within their conditions, and the comments kept among them, as mbse-programs has them. A program's directives are
mbse-programs' to hold, among its declarations; here a directive is held by its own holder, `defined(X)` by C's and
C++'s expressions, and a directive or a comment by an enumeration's body, among its enumerators
([8.07 · C Types](08_07_C_Types.ipynb)). This page builds a directive, then each kind of directive, then a conditional
one with its branches; then what a condition binds, what validation says, where C and C++ read its terms, and the
editions it covers.

Like the rest of Part I, it starts from `dialects()`. `S` is its holder namespace, `amd64` the ABI a C family's
holder names (tutorial 2), `variable` writes a name in C's terms (tutorial 4), and `name` an identifier, the name a
directive defines or tests, each a spec the builders call:

In [1]:
import assert from "node:assert/strict";
import * as C from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as Cpp from "@mbse/expressions/Cpp/Expressions";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { Repr } from "@mbse/schemas/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const amd64 = store.Instances.Expressions.SystemV.AMD64;
const variable = (word: string) => (d: any) => d.variable((v: any) => v.name(word));
const name = (word: string) => (n: any) => n.spelling(word);
const DIALECT = CPreprocessor.Expressions.DIALECT;

/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(evaluate: (expression: any) => unknown, expression: any): string {
  try {
    return String(evaluate(expression));
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.08.1"></a>
## 08.08.1 · A directive, `#define WIDTH 8`

```text
#define WIDTH 8
```

`#define WIDTH 8` defines an object-like macro: a `definedirective`, whose `name` is the identifier it defines, C's
`Identifier` helper, and whose `replacement` is the rest of its line, kept as written. A replacement is a sequence of
tokens, not an expression: the preprocessor substitutes it wherever `WIDTH` is written, before C or C++ parses
anything, so the dialect keeps it as text. Its holder, `S.C.Preprocessor.Expression`, names an ABI as every C family
holder does:

In [2]:
const width = store.transaction(() => S.C.Preprocessor.Expression((e: any) => e.root(
  (u: any) => u.definedirective((d: any) => d.name(name("WIDTH")).replacement("8"))).abi([amd64])));
const term = Terms.term(width);
const form = Terms.form(term);
console.log(CPreprocessor.Text.ToText(width), "|", form.kind, "of", Terms.dialectOf(term));
console.log("its name:", Repr.repr(form.attributes.get("name")), "| its replacement:", form.attributes.get("replacement"));
console.log(DIALECT.name(), DIALECT.tag, "| its kinds:", [...DIALECT.kinds().keys()].join(", "));
assert.equal(CPreprocessor.Text.ToText(width), "#define WIDTH 8");

#define WIDTH 8 | definedirective of Expressions.C.Preprocessor.Term


its name: Part(kind='identifier', attributes={'spelling': 'WIDTH'}, size=0) | its replacement: 8


C Preprocessor c_preprocessor | its kinds: includedirective, definedirective, ifdirective, ifdefdirective, elifdirective, elifdefdirective, elsedirective, otherdirective, definedexpression, comment, error, missing


<a id="T08.08.2"></a>
## 08.08.2 · Each directive

```text
#include <stdio.h>        #include "a.h"        #define MAX(a, b) ((a) > (b) ? (a) : (b))
#define LOG(...) f(__VA_ARGS__)        #undef N        #pragma once        // note
```

Each directive is a kind of its own, with what its grammar writes as properties: an `includedirective`'s `path`, and
whether it is a `system` header, between angle brackets; a function-like `definedirective`'s `parameters`, identifiers,
and whether it is `variadic`; and an `otherdirective` for the directives that hold only their text, `#undef`,
`#pragma`, `#line`, `#error`, named by its `directive`. A comment is a kind too, `comment`, a line comment or a block
one, kept where it was written:

In [3]:
const directives: any[] = store.transaction(() => [
  (u: any) => u.includedirective((d: any) => d.path("stdio.h").system(true)),
  (u: any) => u.includedirective((d: any) => d.path("a.h")),
  (u: any) => u.definedirective((d: any) => d.name(name("MAX")).function_like(true)
    .parameters([name("a"), name("b")]).replacement("((a) > (b) ? (a) : (b))")),
  (u: any) => u.definedirective((d: any) => d.name(name("LOG")).function_like(true).variadic(true)
    .replacement("f(__VA_ARGS__)")),
  (u: any) => u.otherdirective((d: any) => d.directive("undef").text("N")),
  (u: any) => u.otherdirective((d: any) => d.directive("pragma").text("once")),
  (u: any) => u.comment((c: any) => c.text(" note")),
  (u: any) => u.comment((c: any) => c.text(" a block ").block(true)),
].map((spec) => S.C.Preprocessor.Expression((e: any) => e.root(spec).abi([amd64]))));
for (const directive of directives) {
  const form = Terms.form(Terms.term(directive));
  console.log(CPreprocessor.Text.ToText(directive).padEnd(42), form.kind.padEnd(16), [...form.attributes.keys()].join(", "));
}
assert.deepEqual(directives.map((d) => CPreprocessor.Text.ToText(d)), [
  "#include <stdio.h>", '#include "a.h"', "#define MAX(a, b) ((a) > (b) ? (a) : (b))", "#define LOG(...) f(__VA_ARGS__)",
  "#undef N", "#pragma once", "// note", "/* a block */"]);

#include <stdio.h>                         includedirective path, system


#include "a.h"                             includedirective path


#define MAX(a, b) ((a) > (b) ? (a) : (b))  definedirective  name, function_like, parameters, replacement


#define LOG(...) f(__VA_ARGS__)            definedirective  name, function_like, variadic, replacement


#undef N                                   otherdirective   directive, text


#pragma once                               otherdirective   directive, text


// note                                    comment          text


/* a block */                              comment          block, text


<a id="T08.08.3"></a>
## 08.08.3 · A conditional directive and its branches

```text
#if defined(X) && X > 1
A
#elifndef Z
B
#else
// none
#endif
```

A conditional directive includes the lines it holds when its condition holds. `#if`'s `condition` is a term of C's or
C++'s, here C's `&&` of `defined(X)`, the preprocessor's `definedexpression`, which C's `Term` lists among its own kinds,
and `X > 1`. Its `items` are what it includes: enumerators, directives and comments, terms of whichever dialect holds
them, `C Types`' `enumerator` here. Its `alternative` is the branch after it, `#elif`, `#elifdef`, `#elifndef` (an
`elifdefdirective` whose `negated` is set) or `#else`, each holding the next, so the chain is one tree, which the
printer closes with one `#endif`. A form gives the condition and the next branch as its subterms, then the items:

In [4]:
const enumerator = (word: string) => (t: any) => t.c_types((u: any) => u.enumerator((r: any) => r.name(name(word))));
const one = (t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(1n)));
const condition = (t: any) => t.c((d: any) => d.binaryexpression((b: any) => b
  .left((l: any) => l.definedexpression((x: any) => x.name(name("X")))).operator("&&")
  .right((r: any) => r.binaryexpression((g: any) => g.left(variable("X")).operator(">").right(one)))));
const conditional = store.transaction(() => S.C.Preprocessor.Expression((e: any) => e.root((u: any) => u.ifdirective((d: any) => d
  .condition(condition).items([enumerator("A")])
  .alternative((a: any) => a.elifdefdirective((f: any) => f.negated(true).name(name("Z")).items([enumerator("B")])
    .alternative((a: any) => a.elsedirective((s: any) => s.items([
      (t: any) => t.c_preprocessor((u: any) => u.comment((c: any) => c.text(" none")))]))))))).abi([amd64])));
console.log(CPreprocessor.Text.ToText(conditional));
for (const t of Terms.form(Terms.term(conditional)).arguments) {
  console.log(" ", Terms.kindOf(t)[0].padEnd(16), `of ${DIALECT.owner(t).name().padEnd(15)}`,
    CPreprocessor.Text.ToText(t).replaceAll("\n", " | "));
}
assert.equal(CPreprocessor.Text.ToText(conditional), "#if defined(X) && X > 1\nA\n#elifndef Z\nB\n#else\n// none\n#endif");

#if defined(X) && X > 1
A
#elifndef Z
B
#else
// none
#endif


  binaryexpression of C               defined(X) && X > 1


  elifdefdirective of C Preprocessor  #elifndef Z | B | #else | // none | #endif


  enumerator       of C Types         A


<a id="T08.08.4"></a>
## 08.08.4 · What a condition binds, and what is checked

A condition's names are macros' names, which the preprocessor replaces before it evaluates the condition: they are
not a term's references, so a conditional directive binds them within its condition, and `X` needs nothing from its
scope (tutorial 7). It binds them nowhere else: an enumerator its branch holds, whose value reads `X`, reads a
variable, which its scope must give. `macros` gives the names a condition reads as macros, `defined(X)`'s among them,
whose name is a macro's, never a reference:

In [5]:
const valued = store.transaction(() => S.C.Preprocessor.Expression((e: any) => e.root((u: any) => u.ifdirective((d: any) => d.condition(condition)
  .items([(t: any) => t.c_types((u: any) => u.enumerator((r: any) => r.name(name("A")).value(variable("X"))))]))).abi([amd64])));
console.log("macros:", [...CPreprocessor.Expressions.macros(Terms.form(Terms.term(conditional)).arguments[0])].sort().join(", "));
for (const directive of [conditional, valued]) {
  console.log(CPreprocessor.Text.ToText(directive).replaceAll("\n", " | "), "|", DIALECT.validate(directive).join("; ") || "valid",
    "| needs:", [...Symbolics.free(directive, DIALECT)].sort().join(", ") || "nothing");
}
assert.deepEqual(DIALECT.validate(conditional), []);
assert.deepEqual(DIALECT.validate(valued), ["argument 0: value: variable 'X' is not bound"]);

macros: X


#if defined(X) && X > 1 | A | #elifndef Z | B | #else | // none | #endif | valid | needs: nothing


#if defined(X) && X > 1 | A = X | #endif | argument 0: value: variable 'X' is not bound | needs: X


Validation checks each directive as its grammar has it, in mbse-programs' words: an `#include` names a path or a macro,
a directive's names are identifiers, a branch after another is an `#elif`, an `#elifdef` or an `#else`. Text that does
not parse is kept as written, as `error` ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)),
printed back and reported:

In [6]:
const wrong: Record<string, any> = store.transaction(() => Object.fromEntries(([
  ["#include", (u: any) => u.includedirective((d: any) => d)],
  ["#define 1A", (u: any) => u.definedirective((d: any) => d.name(name("1A")))],
  ["#ifdef A ... #ifdef B", (u: any) => u.ifdefdirective((d: any) => d.name(name("A"))
    .alternative((a: any) => a.ifdefdirective((f: any) => f.name(name("B")))))],
  ["#if (", (u: any) => u.error((r: any) => r.text("#if ("))],
] as [string, unknown][]).map(([source, spec]) => [source, S.C.Preprocessor.Expression((e: any) => e.root(spec).abi([amd64]))])));
for (const [source, directive] of Object.entries(wrong)) {
  console.log(source.padEnd(22), "|", DIALECT.validate(directive).join("; "));
}
assert.deepEqual(DIALECT.validate(wrong["#include"]), ["an IncludeDirective has a path or a macro"]);
assert.equal(CPreprocessor.Text.ToText(wrong["#if ("]), "#if (");

#include               | an IncludeDirective has a path or a macro


#define 1A             | '1A' is not an identifier


#ifdef A ... #ifdef B  | an IfdefDirective's alternative is an elifdirective, an elifdefdirective or an elsedirective


#if (                  | invalid syntax '#if ('


<a id="T08.08.5"></a>
## 08.08.5 · Read by C and C++, and by their types

```text
defined(A) || defined(B)        enum E {
                                    A,
                                #ifdef WIDE
                                    B,
                                #endif
                                    C
                                }
```

C and C++ read `C Preprocessor`'s terms where their grammars put them. `definedexpression` stands among C's and C++'s
own kinds, so `defined(A) || defined(B)` is a C expression, or a C++ one, whose operands are the preprocessor's terms;
and its directives and comment stand among `C Types`' and `Cpp Types`' kinds, so an enumeration's body holds them
between its enumerators, printed at the start of their lines by the family's one printer (tutorial 8, 08.5). Neither
evaluates them: a directive is the preprocessor's to carry out, before evaluation, and each family's evaluator refuses
one by its dialect's name, wherever it stands:

In [7]:
const defined = (word: string) => (d: any) => d.definedexpression((x: any) => x.name(name(word)));
const [either, body]: [Record<string, [any, any]>, any] = store.transaction(() => [
  {
    C: [C, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.binaryexpression((b: any) => b
      .left(defined("A")).operator("||").right(defined("B")))).abi([amd64]))],
    Cpp: [Cpp, S.Cpp.Expressions.Expression((e: any) => e.root((u: any) => u.binaryexpression((b: any) => b
      .left(defined("A")).operator("||").right(defined("B")))).abi([amd64]))],
  },
  S.C.Types.Expression((e: any) => e.root((u: any) => u.typeid((i: any) => i.specifiers([
    (s: any) => s.enumspecifier((n: any) => n.key("enum").name((m: any) => m.identifier(name("E"))).body(true).enumerators([
      (t: any) => t.enumerator((r: any) => r.name(name("A"))),
      (t: any) => t.ifdefdirective((d: any) => d.name(name("WIDE")).items([enumerator("B")])),
      (t: any) => t.enumerator((r: any) => r.name(name("C")))]))]))).abi([amd64])),
]);
for (const [language, [family, expression]] of Object.entries(either)) {
  const operands = Terms.form(Terms.term(expression)).arguments;
  console.log(language.padEnd(4), family.Text.ToText(expression), "|", operands.map(
    (a: any) => `${Terms.kindOf(a)[0]} of ${family.Expressions.DIALECT.owner(a).name()}`).join(", "),
    "|", refused((x) => family.Evaluators.OfAny(x), expression));
}
console.log(CTypes.Text.ToText(body));
const directive = Terms.form(Terms.term(body)).arguments[2];
console.log(Terms.kindOf(directive)[0], "of", CTypes.Expressions.DIALECT.owner(directive).name(), "|",
  refused((x) => C.Evaluators.OfAny(x), conditional));
assert.ok(C.Text.ToText(either.C![1]) === "defined(A) || defined(B)" && Cpp.Text.ToText(either.Cpp![1]) === "defined(A) || defined(B)");
assert.equal(CTypes.Text.ToText(body), "enum E {\n    A,\n#ifdef WIDE\n    B,\n#endif\n    C\n}");

C    defined(A) || defined(B) | definedexpression of C Preprocessor, definedexpression of C Preprocessor | C Preprocessor definedexpression is not evaluated


Cpp  defined(A) || defined(B) | definedexpression of C Preprocessor, definedexpression of C Preprocessor | C Preprocessor definedexpression is not evaluated


enum E {
    A,
#ifdef WIDE
    B,
#endif
    C
}


ifdefdirective of C Preprocessor | C Preprocessor ifdirective is not evaluated


<a id="T08.08.6"></a>
## 08.08.6 · Its editions

One dialect for two languages covers both languages' editions, C's three and C++'s three, each by its designation and
its common name ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)).
Every kind is in all six but one: `#elifdef` and `#elifndef` came with C23 and C++23, so `elifdefdirective` is recorded in
those two alone, and `Standards.of` reads, for a kind, the editions that have it:

In [8]:
const pre = store.Instances.Expressions.C.Preprocessor; // the dialect, by its global name
console.log([...Standards.standards(pre)].map(([designation, edition]) => `${designation} (${Standards.tags(edition)[1]})`).join(", "));
const kinds = new Map<string, any>(store.registered(DIALECT.union).branches.map((b: any) => [b.name, b.schema]));
for (const kind of ["ifdirective", "elifdefdirective"]) {
  console.log(kind.padEnd(16), [...Standards.of(pre, kinds.get(kind)).values()].map((edition) => Standards.tags(edition)[1]).join(", "));
}
assert.deepEqual([...Standards.of(pre, kinds.get("elifdefdirective")).values()].map((e) => Standards.tags(e)[1]), ["C23", "C++23"]);

ISO/IEC 9899:2011 (C11), ISO/IEC 9899:2018 (C17), ISO/IEC 9899:2024 (C23), ISO/IEC 14882:2017 (C++17), ISO/IEC 14882:2020 (C++20), ISO/IEC 14882:2024 (C++23)


ifdirective      C11, C17, C23, C++17, C++20, C++23


elifdefdirective C23, C++23


## What you learned

- **The preprocessor is one dialect for C and C++**, `C Preprocessor`: its directives, `defined(X)` and comments, each
  a kind with what its grammar writes, a macro's replacement kept as written.
- **A conditional directive is one tree**: its condition a C or C++ term, its items any dialect's terms, each branch
  after it its `alternative`.
- **A condition binds its macros' names**, within itself alone.
- **C and C++ read its terms where their grammars put them**: `defined(X)` among their expressions, directives and
  comments among their enumerators; none is evaluated.
- **Its editions are both languages'**, `#elifdef` C23's and C++23's.

Next, C++'s own type-ids, [8.09 · Cpp Types](08_09_Cpp_Types.ipynb).

<!-- nav -->
[← 8 · C Types](08_07_C_Types.ipynb) · [Home](../../README.md) · [8.09 · Cpp Types →](08_09_Cpp_Types.ipynb)